# REST APIs: Flask & FastAPI

Build HTTP APIs with request validation, error handling, and automatic documentation.

**Install:** `pip install flask fastapi uvicorn` | **Use:** Web services, microservices, JSON APIs

## Flask: Lightweight Framework

In [ ]:
from flask import Flask, request, jsonify

app = Flask(__name__)
users = {1: {"name": "Alice", "email": "alice@example.com"}}

@app.route('/users/<int:user_id>', methods=['GET'])
def get_user(user_id):
    user = users.get(user_id)
    if not user:
        return jsonify({"error": "User not found"}), 404
    return jsonify(user)

@app.route('/users', methods=['POST'])
def create_user():
    data = request.json
    user_id = max(users.keys()) + 1
    users[user_id] = data
    return jsonify({"id": user_id, **data}), 201

@app.route('/users/<int:user_id>', methods=['PUT'])
def update_user(user_id):
    if user_id not in users:
        return jsonify({"error": "User not found"}), 404
    users[user_id].update(request.json)
    return jsonify(users[user_id])

@app.route('/users/<int:user_id>', methods=['DELETE'])
def delete_user(user_id):
    if user_id not in users:
        return jsonify({"error": "User not found"}), 404
    del users[user_id]
    return jsonify({"message": "User deleted"})

print("Flask app routes defined")

## FastAPI: Modern Framework with Auto-Docs

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

fastapi_app = FastAPI()

class User(BaseModel):
    name: str
    email: str

class UserInDB(User):
    id: int

db = {1: {"id": 1, "name": "Alice", "email": "alice@example.com"}}

@fastapi_app.get("/users/{user_id}", response_model=UserInDB)
async def get_user(user_id: int):
    if user_id not in db:
        raise HTTPException(status_code=404, detail="User not found")
    return db[user_id]

@fastapi_app.post("/users", response_model=UserInDB, status_code=201)
async def create_user(user: User):
    user_id = max(db.keys()) + 1 if db else 1
    db[user_id] = {"id": user_id, **user.model_dump()}
    return db[user_id]

@fastapi_app.delete("/users/{user_id}")
async def delete_user(user_id: int):
    if user_id not in db:
        raise HTTPException(status_code=404, detail="User not found")
    del db[user_id]
    return {"message": "User deleted"}

print("FastAPI app defined")

## Testing APIs with TestClient

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(fastapi_app)

response = client.get("/users/1")
print("GET /users/1:", response.json())

new_user = {"name": "Bob", "email": "bob@example.com"}
response = client.post("/users", json=new_user)
print("POST /users:", response.json())

response = client.delete("/users/2")
print("DELETE /users/2:", response.json())

## Request Validation with Pydantic

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, EmailStr, Field

validation_app = FastAPI()

class ValidatedUser(BaseModel):
    name: str = Field(..., min_length=1, max_length=100)
    email: EmailStr
    age: int = Field(..., ge=18, le=120)

@validation_app.post("/register")
async def register(user: ValidatedUser):
    return {"status": "registered", "user": user.model_dump()}

client = TestClient(validation_app)
valid_request = {"name": "Charlie", "email": "charlie@example.com", "age": 25}
print("Valid:", client.post("/register", json=valid_request).json())

invalid_email = {"name": "Charlie", "email": "not-an-email", "age": 25}
response = client.post("/register", json=invalid_email)
print("Invalid email status:", response.status_code)

## Error Handling

In [ ]:
from fastapi import FastAPI, HTTPException

error_app = FastAPI()

@error_app.get("/items/{item_id}")
async def get_item(item_id: int):
    if item_id < 0:
        raise HTTPException(status_code=400, detail="Item ID must be positive")
    if item_id > 1000:
        raise HTTPException(status_code=404, detail="Item not found")
    return {"id": item_id, "price": 10.99}

client = TestClient(error_app)
response = client.get("/items/999")
print("Error response:", response.json())
print("Status code:", response.status_code)

---

## What is UV?
**UV** is a fast, modern Python package manager (faster than pip):
- Written in Rust (significantly faster)
- `uv pip install package` replaces `pip install package`
- `uv run script.py` runs scripts without activating venv
- `uv venv` creates virtual environments
- No need for `source venv/bin/activate` (Unix) or `venv\Scripts\activate` (Windows)
- Example: `uv run --with flask --with fastapi python app.py`

## What is Uvicorn?
**Uvicorn** is an ASGI web server for running FastAPI (and async Python web apps):
- FastAPI needs Uvicorn to run (Flask uses its own built-in server)
- Supports async/await: `async def get_users():`
- Command: `uvicorn app:app --reload --port 8000`
- Auto-reloads on code changes (with `--reload` flag)
- Why separate? Uvicorn handles async I/O efficiently; Flask server doesn't

---

## How to Run & Call APIs

### Setup: Using Python or UV
```bash
# With traditional Python + pip:
python -m venv venv
source venv/bin/activate  # Windows: venv\Scripts\activate
pip install flask fastapi uvicorn requests

# With UV (faster, simpler):
uv venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
uv pip install flask fastapi uvicorn requests
# OR direct: uv run --with flask app.py
```

### Run Flask Server (Default & Custom Ports)
```bash
# Default port 5000:
python app.py
uv run app.py

# Custom port 8080 (modify app.py):
# app.run(host='0.0.0.0', port=8080, debug=True)
python app.py
uv run app.py

# Custom port 8080 via environment:
export FLASK_PORT=8080  # Windows: set FLASK_PORT=8080
python app.py
FLASK_PORT=8080 uv run app.py
```

### Run FastAPI Server (Default & Custom Ports)
```bash
# Default port 8000:
uvicorn app:app --reload
uv run --with fastapi --with uvicorn uvicorn app:app --reload

# Custom port 3000:
uvicorn app:app --reload --port 3000 --host 0.0.0.0
uv run --with fastapi --with uvicorn uvicorn app:app --reload --port 3000

# Custom port 9000:
uvicorn app:app --reload --port 9000
uv run --with fastapi --with uvicorn uvicorn app:app --reload --port 9000

# Docs: http://localhost:8000/docs (default) or http://localhost:3000/docs
```

### Call APIs from Python
```python
import requests

BASE_URL = 'http://localhost:5000'  # Flask (change port as needed)
# BASE_URL = 'http://localhost:8000'  # FastAPI (change port as needed)

# GET
response = requests.get(f'{BASE_URL}/users/1')
print(response.json())

# POST
payload = {"name": "Bob", "email": "bob@example.com"}
response = requests.post(f'{BASE_URL}/users', json=payload)
print(response.json())

# PUT (Flask only)
updates = {"name": "Bob Updated", "email": "bob.new@example.com"}
response = requests.put(f'{BASE_URL}/users/1', json=updates)

# DELETE
response = requests.delete(f'{BASE_URL}/users/1')
print(response.status_code)
```

### Test in Jupyter (No Server)
```python
from fastapi.testclient import TestClient

client = TestClient(fastapi_app)

# GET
response = client.get("/users/1")
print(response.json())

# POST
payload = {"name": "Bob", "email": "bob@example.com"}
response = client.post("/users", json=payload)
print(response.json())

# DELETE
response = client.delete("/users/1")
print(response.status_code)
```

### HTTP Status Codes
- `200` OK
- `201` Created
- `204` No Content
- `400` Bad Request (validation error)
- `404` Not Found
- `422` Unprocessable Entity (validation error)
- `500` Internal Server Error

## Resources

- [Flask Documentation](https://flask.palletsprojects.com/)
- [FastAPI Documentation](https://fastapi.tiangolo.com/)
- [REST API Design](https://restfulapi.net/)
